In [ ]:
pip install pandas numpy sentence-transformers transformers torch scikit-learn


In [13]:
import pandas as pd

# Load data
df = pd.read_csv("/content/tickets_model_input_df.csv")

df

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5676,NaN,I am seeking detailed instructions for integra...,Thank you for your interest in integrating Sma...,Request,IT Support,medium,en,52.0,Feature,Documentation,Support,Productivity,Integration,Workflow,NaN,NaN
5677,Intermittent Connectivity Problems Detected Ac...,We are experiencing sporadic connectivity disr...,<name> apologizes for the intermittent connect...,Incident,General Inquiry,low,en,52.0,Network,Disruption,Performance,Outage,NaN,NaN,NaN,NaN
5678,Request for Advanced Analytics Assistance,Customer Support is reaching out to inquire ab...,Please provide comprehensive documentation for...,Request,Technical Support,high,en,52.0,Feature,Documentation,Training,Support,NaN,NaN,NaN,NaN
5679,NaN,"Customer Support, I am reaching out to inquire...","<name>, thank you for contacting the customer ...",Request,Technical Support,medium,en,52.0,Feature,Product,Documentation,Feedback,IT,Tech Support,NaN,NaN


In [14]:
# Handle missing values
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")
df["answer"] = df["answer"].fillna("")

# Combine text
df["full_text"] = df["subject"] + " " + df["body"] + " " + df["answer"]

print(df.head())

                                             subject  \
0                                 Account Disruption   
1  Query About Smart Home System Integration Feat...   
2                  Inquiry Regarding Invoice Details   
3  Question About Marketing Agency Software Compa...   
4                                      Feature Query   

                                                body  \
0  Dear Customer Support Team,\n\nI am writing to...   
1  Dear Customer Support Team,\n\nI hope this mes...   
2  Dear Customer Support Team,\n\nI hope this mes...   
3  Dear Support Team,\n\nI hope this message reac...   
4  Dear Customer Support,\n\nI hope this message ...   

                                              answer      type  \
0  Thank you for reaching out, <name>. We are awa...  Incident   
1  Thank you for your inquiry. Our products suppo...   Request   
2  We appreciate you reaching out with your billi...   Request   
3  Thank you for your inquiry. Our product suppor...   Problem

In [15]:
import re
# data cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # remove urls
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["full_text"].apply(clean_text)
df

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,full_text,clean_text
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,"Account Disruption Dear Customer Support Team,...","account disruption dear customer support team,..."
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN,"Feature Query Dear Customer Support,\n\nI hope...","feature query dear customer support,\n\ni hope..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5676,,I am seeking detailed instructions for integra...,Thank you for your interest in integrating Sma...,Request,IT Support,medium,en,52.0,Feature,Documentation,Support,Productivity,Integration,Workflow,NaN,NaN,I am seeking detailed instructions for integr...,i am seeking detailed instructions for integra...
5677,Intermittent Connectivity Problems Detected Ac...,We are experiencing sporadic connectivity disr...,<name> apologizes for the intermittent connect...,Incident,General Inquiry,low,en,52.0,Network,Disruption,Performance,Outage,NaN,NaN,NaN,NaN,Intermittent Connectivity Problems Detected Ac...,intermittent connectivity problems detected ac...
5678,Request for Advanced Analytics Assistance,Customer Support is reaching out to inquire ab...,Please provide comprehensive documentation for...,Request,Technical Support,high,en,52.0,Feature,Documentation,Training,Support,NaN,NaN,NaN,NaN,Request for Advanced Analytics Assistance Cust...,request for advanced analytics assistance cust...
5679,,"Customer Support, I am reaching out to inquire...","<name>, thank you for contacting the customer ...",Request,Technical Support,medium,en,52.0,Feature,Product,Documentation,Feedback,IT,Tech Support,NaN,NaN,"Customer Support, I am reaching out to inquir...","customer support, i am reaching out to inquire..."


In [38]:
# Perfomed vectorizer & use stop words (get rid of unwanted words in our tokens/keys)
from sklearn.feature_extraction.text import CountVectorizer

from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
stop_words = stopwords.words("english")
stop_words = stop_words + ["regards", "you", "and", "thanks", "urgent","please","provide"]
vectorizer_model = CountVectorizer(
    stop_words = stop_words,
    ngram_range = (1,2),
    min_df = 5
)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [41]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [42]:
from bertopic import BERTopic
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model = vectorizer_model,
    verbose=True,
    nr_topics = 50
)

In [43]:
topics, probabilities = topic_model.fit_transform(df["clean_text"].tolist())

df["topic"] = topics

2026-02-19 07:14:33,028 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/178 [00:00<?, ?it/s]

2026-02-19 07:22:14,300 - BERTopic - Embedding - Completed ✓
2026-02-19 07:22:14,301 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-19 07:22:23,378 - BERTopic - Dimensionality - Completed ✓
2026-02-19 07:22:23,380 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-19 07:22:23,573 - BERTopic - Cluster - Completed ✓
2026-02-19 07:22:23,574 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-02-19 07:22:24,610 - BERTopic - Representation - Completed ✓
2026-02-19 07:22:24,612 - BERTopic - Topic reduction - Reducing number of topics
2026-02-19 07:22:24,644 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-19 07:22:25,682 - BERTopic - Representation - Completed ✓
2026-02-19 07:22:25,687 - BERTopic - Topic reduction - Reduced number of topics from 99 to 50


In [44]:
topic_model.visualize_topics()

In [47]:
def generate_tag(topic_id):
    words = topic_model.get_topic(topic_id)
    if words:
        return "_".join([word for word, _ in words[:3]])
    return "noise"

df["auto_tag"] = df["topic"].apply(generate_tag)

print(df[["clean_text", "auto_tag"]].head())


                                          clean_text  \
0  account disruption dear customer support team,...   
1  query about smart home system integration feat...   
2  inquiry regarding invoice details dear custome...   
3  question about marketing agency software compa...   
4  feature query dear customer support,\n\ni hope...   

                                 auto_tag  
0  account management_account_centralized  
1               issue_problem_integration  
2           billing_payment_discrepancies  
3               teams_agile_collaboration  
4              marketing_services_ni hope  


In [48]:
df["auto_tag"].unique()

array(['account management_account_centralized',
       'issue_problem_integration', 'billing_payment_discrepancies',
       'teams_agile_collaboration', 'marketing_services_ni hope',
       'devices_network_connectivity', 'saas_saas platform_platform',
       'python_machine learning_machine', 'security_data_protocols',
       'login_employee_intermittent', 'payment_plans_billing',
       'security_healthcare_access', 'integration_step_documentation',
       'investment_analytics_data analytics', 'return_exchange_order',
       'saas_project management_scalability', 'agile_team_collaboration',
       'synchronization_data synchronization_failures',
       'video_firmware_monitor', 'investment_analytics_data',
       'dashboard_project_dashboards', 'agile_kubernetes_architecture',
       'digital_agency_marketing agency', 'integrations_recent_update',
       'brand_digital_strategies', 'outages_server_outage',
       'smartsheet_step_integrating', 'quickbooks_step_quickbooks online',
 

In [49]:
df

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,full_text,clean_text,topic,auto_tag
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN,"Account Disruption Dear Customer Support Team,...","account disruption dear customer support team,...",40,account management_account_centralized
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN,Query About Smart Home System Integration Feat...,query about smart home system integration feat...,-1,issue_problem_integration
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...,9,billing_payment_discrepancies
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN,Question About Marketing Agency Software Compa...,question about marketing agency software compa...,21,teams_agile_collaboration
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Thank you for your inquiry. Please specify whi...,Request,Technical Support,high,en,51.0,Feature,Product,Documentation,Feedback,NaN,NaN,NaN,NaN,"Feature Query Dear Customer Support,\n\nI hope...","feature query dear customer support,\n\ni hope...",18,marketing_services_ni hope
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5676,,I am seeking detailed instructions for integra...,Thank you for your interest in integrating Sma...,Request,IT Support,medium,en,52.0,Feature,Documentation,Support,Productivity,Integration,Workflow,NaN,NaN,I am seeking detailed instructions for integr...,i am seeking detailed instructions for integra...,20,smartsheet_step_integrating
5677,Intermittent Connectivity Problems Detected Ac...,We are experiencing sporadic connectivity disr...,<name> apologizes for the intermittent connect...,Incident,General Inquiry,low,en,52.0,Network,Disruption,Performance,Outage,NaN,NaN,NaN,NaN,Intermittent Connectivity Problems Detected Ac...,intermittent connectivity problems detected ac...,11,devices_network_connectivity
5678,Request for Advanced Analytics Assistance,Customer Support is reaching out to inquire ab...,Please provide comprehensive documentation for...,Request,Technical Support,high,en,52.0,Feature,Documentation,Training,Support,NaN,NaN,NaN,NaN,Request for Advanced Analytics Assistance Cust...,request for advanced analytics assistance cust...,4,investment_analytics_data analytics
5679,,"Customer Support, I am reaching out to inquire...","<name>, thank you for contacting the customer ...",Request,Technical Support,medium,en,52.0,Feature,Product,Documentation,Feedback,IT,Tech Support,NaN,NaN,"Customer Support, I am reaching out to inquir...","customer support, i am reaching out to inquire...",-1,issue_problem_integration
